In [1]:
from __future__ import annotations

import json
import os
from getpass import getpass

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery, Filter

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
COLLECTION_NAME = "SupportTicket"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

tickets = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="body",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="customer_type",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="ai_summary",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="ai_category",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="ai_priority",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="ai_action",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: SupportTicket


In [5]:
support_tickets = [
    {
        "title": "OpenAI key does not work in Weaviate Cloud",
        "body": (
            "I connected to my Weaviate Cloud cluster, but when I try to insert data "
            "with text2vec_openai, I get a 401 error from OpenAI saying the API key is incorrect."
        ),
        "customer_type": "developer",
    },
    {
        "title": "Hybrid search returns unexpected results",
        "body": (
            "I am using hybrid search with alpha set to 0.5. Some results are relevant, "
            "but others look too keyword-based. I want to understand how alpha affects ranking."
        ),
        "customer_type": "developer",
    },
    {
        "title": "Need help designing RAG over project notebooks",
        "body": (
            "I have many Jupyter notebooks and Markdown notes about Weaviate. "
            "I want to build a focused RAG collection that retrieves only useful chunks."
        ),
        "customer_type": "data_scientist",
    },
    {
        "title": "Images are not searchable in Weaviate Cloud",
        "body": (
            "I migrated from local Docker to Weaviate Cloud. Previously I used multi2vec_clip. "
            "Now I want to search images using local CLIP embeddings and self-provided vectors."
        ),
        "customer_type": "ml_engineer",
    },
    {
        "title": "How to isolate data for multiple companies",
        "body": (
            "I am building a SaaS product and want each customer company to have isolated data. "
            "I heard that Weaviate supports multi-tenancy but I am not sure how to use it."
        ),
        "customer_type": "product_engineer",
    },
    {
        "title": "Objects are inserted but I do not understand UUIDs",
        "body": (
            "I inserted objects into a Weaviate collection and received UUIDs. "
            "I want to understand how to fetch, update and delete objects using those UUIDs."
        ),
        "customer_type": "beginner",
    },
]

In [6]:
result = tickets.data.insert_many(support_tickets)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [7]:
response = tickets.query.fetch_objects(
    limit=10,
    return_properties=[
        "title",
        "body",
        "customer_type",
        "ai_summary",
        "ai_category",
        "ai_priority",
        "ai_action",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Title:", obj.properties["title"])
    print("Customer type:", obj.properties["customer_type"])
    print("AI summary:", obj.properties.get("ai_summary"))
    print("AI category:", obj.properties.get("ai_category"))
    print("-" * 80)

UUID: 3b186d20-ea98-4e06-bd9b-071020252d08
Title: Objects are inserted but I do not understand UUIDs
Customer type: beginner
AI summary: None
AI category: None
--------------------------------------------------------------------------------
UUID: 51d7c3a9-61be-4698-9fdf-67e48de590a4
Title: OpenAI key does not work in Weaviate Cloud
Customer type: developer
AI summary: None
AI category: None
--------------------------------------------------------------------------------
UUID: 7723a2b5-da4b-4a22-826a-d4668b369a8c
Title: Images are not searchable in Weaviate Cloud
Customer type: ml_engineer
AI summary: None
AI category: None
--------------------------------------------------------------------------------
UUID: 7ce6c4ab-714d-4cc1-8212-e720ad5d61a4
Title: How to isolate data for multiple companies
Customer type: product_engineer
AI summary: None
AI category: None
--------------------------------------------------------------------------------
UUID: 9c31b0de-2c2e-45de-9162-de0ec9a39827
Titl

In [8]:
response = tickets.query.near_text(
    query="problem with OpenAI API key and Weaviate Cloud connection",
    limit=3,
    return_properties=[
        "title",
        "body",
        "customer_type",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Customer type:", obj.properties["customer_type"])
    print("Body:", obj.properties["body"])
    print("-" * 80)

Distance: 0.20972776412963867
Title: OpenAI key does not work in Weaviate Cloud
Customer type: developer
Body: I connected to my Weaviate Cloud cluster, but when I try to insert data with text2vec_openai, I get a 401 error from OpenAI saying the API key is incorrect.
--------------------------------------------------------------------------------
Distance: 0.5269652605056763
Title: Images are not searchable in Weaviate Cloud
Customer type: ml_engineer
Body: I migrated from local Docker to Weaviate Cloud. Previously I used multi2vec_clip. Now I want to search images using local CLIP embeddings and self-provided vectors.
--------------------------------------------------------------------------------
Distance: 0.6040221452713013
Title: How to isolate data for multiple companies
Customer type: product_engineer
Body: I am building a SaaS product and want each customer company to have isolated data. I heard that Weaviate supports multi-tenancy but I am not sure how to use it.
--------------

In [9]:
response = tickets.generate.near_text(
    query="Weaviate support tickets about search, cloud, RAG, vectors and object management",
    limit=10,
    single_prompt=(
        "You are a technical support classifier for Weaviate. "
        "Analyze this support ticket and return only valid JSON with these keys: "
        "summary, category, priority, action. "
        "Allowed categories: authentication, hybrid_search, rag, image_search, multi_tenancy, crud, other. "
        "Allowed priorities: low, medium, high. "
        "The action should be one short practical recommendation. "
        "Ticket title: {title}. "
        "Ticket body: {body}. "
        "Customer type: {customer_type}."
    ),
    return_properties=[
        "title",
        "body",
        "customer_type",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Title:", obj.properties["title"])
    print("Generated:")
    print(obj.generated)
    print("-" * 80)

UUID: 7723a2b5-da4b-4a22-826a-d4668b369a8c
Title: Images are not searchable in Weaviate Cloud
Generated:
```json
{
  "summary": "Issues with searching images after migrating to Weaviate Cloud.",
  "category": "image_search",
  "priority": "medium",
  "action": "Ensure that the CLIP embeddings are properly indexed in Weaviate Cloud."
}
```
--------------------------------------------------------------------------------
UUID: c6d13824-2c19-4839-b314-0633999538ac
Title: Need help designing RAG over project notebooks
Generated:
```json
{
  "summary": "Assistance required for designing a RAG collection from Jupyter notebooks and Markdown notes.",
  "category": "rag",
  "priority": "medium",
  "action": "Consider organizing your notes into categories to enhance chunk retrieval."
}
```
--------------------------------------------------------------------------------
UUID: 7ce6c4ab-714d-4cc1-8212-e720ad5d61a4
Title: How to isolate data for multiple companies
Generated:
```json
{
  "summary": "C

In [10]:
def clean_json_response(value: str) -> str:
    cleaned = value.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned.removeprefix("```json").strip()

    if cleaned.startswith("```"):
        cleaned = cleaned.removeprefix("```").strip()

    if cleaned.endswith("```"):
        cleaned = cleaned.removesuffix("```").strip()

    return cleaned

In [11]:
enriched_count = 0

for obj in response.objects:
    raw_generated = obj.generated

    if raw_generated is None:
        print("No generated output for:", obj.uuid)
        continue

    cleaned = clean_json_response(raw_generated)

    try:
        enrichment = json.loads(cleaned)
    except json.JSONDecodeError:
        print("Could not parse generated JSON for:", obj.uuid)
        print(cleaned)
        print("-" * 80)
        continue

    tickets.data.update(
        uuid=obj.uuid,
        properties={
            "ai_summary": enrichment["summary"],
            "ai_category": enrichment["category"],
            "ai_priority": enrichment["priority"],
            "ai_action": enrichment["action"],
        },
    )

    enriched_count += 1

print("Updated objects:", enriched_count)

Updated objects: 6


In [12]:
response = tickets.query.fetch_objects(
    limit=10,
    return_properties=[
        "title",
        "customer_type",
        "ai_summary",
        "ai_category",
        "ai_priority",
        "ai_action",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Title:", obj.properties["title"])
    print("Customer type:", obj.properties["customer_type"])
    print("AI summary:", obj.properties["ai_summary"])
    print("AI category:", obj.properties["ai_category"])
    print("AI priority:", obj.properties["ai_priority"])
    print("AI action:", obj.properties["ai_action"])
    print("-" * 80)

UUID: 3b186d20-ea98-4e06-bd9b-071020252d08
Title: Objects are inserted but I do not understand UUIDs
Customer type: beginner
AI summary: User is unclear on how to use UUIDs for fetching, updating, and deleting objects in Weaviate.
AI category: crud
AI priority: medium
AI action: Refer to the Weaviate documentation on CRUD operations using UUIDs.
--------------------------------------------------------------------------------
UUID: 51d7c3a9-61be-4698-9fdf-67e48de590a4
Title: OpenAI key does not work in Weaviate Cloud
Customer type: developer
AI summary: User is experiencing a 401 error when using OpenAI API key with Weaviate Cloud for data insertion.
AI category: other
AI priority: high
AI action: Check if the OpenAI API key is correctly configured and has the necessary permissions.
--------------------------------------------------------------------------------
UUID: 7723a2b5-da4b-4a22-826a-d4668b369a8c
Title: Images are not searchable in Weaviate Cloud
Customer type: ml_engineer
AI su

In [13]:
response = tickets.query.near_text(
    query="customer has authentication problem with OpenAI API key",
    limit=3,
    return_properties=[
        "title",
        "body",
        "ai_summary",
        "ai_category",
        "ai_priority",
        "ai_action",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Category:", obj.properties["ai_category"])
    print("Priority:", obj.properties["ai_priority"])
    print("Summary:", obj.properties["ai_summary"])
    print("Action:", obj.properties["ai_action"])
    print("-" * 80)

Distance: 0.37547850608825684
Title: OpenAI key does not work in Weaviate Cloud
Category: other
Priority: high
Summary: User is experiencing a 401 error when using OpenAI API key with Weaviate Cloud for data insertion.
Action: Check if the OpenAI API key is correctly configured and has the necessary permissions.
--------------------------------------------------------------------------------
Distance: 0.7412779927253723
Title: Hybrid search returns unexpected results
Category: hybrid_search
Priority: medium
Summary: Customer is experiencing unexpected results from hybrid search with alpha set to 0.5, noting a mix of relevant and keyword-based results.
Action: Review the impact of the alpha parameter on ranking and adjust accordingly.
--------------------------------------------------------------------------------
Distance: 0.7438428401947021
Title: Images are not searchable in Weaviate Cloud
Category: image_search
Priority: medium
Summary: Issues with searching images after migrating t

In [14]:
response = tickets.query.fetch_objects(
    filters=Filter.by_property("ai_category").equal("rag"),
    limit=10,
    return_properties=[
        "title",
        "customer_type",
        "ai_summary",
        "ai_category",
        "ai_priority",
        "ai_action",
    ],
)

for obj in response.objects:
    print("Title:", obj.properties["title"])
    print("Category:", obj.properties["ai_category"])
    print("Priority:", obj.properties["ai_priority"])
    print("Summary:", obj.properties["ai_summary"])
    print("Action:", obj.properties["ai_action"])
    print("-" * 80)

Title: Need help designing RAG over project notebooks
Category: rag
Priority: medium
Summary: Assistance required for designing a RAG collection from Jupyter notebooks and Markdown notes.
Action: Consider organizing your notes into categories to enhance chunk retrieval.
--------------------------------------------------------------------------------


In [15]:
response = tickets.query.fetch_objects(
    filters=Filter.by_property("ai_priority").equal("high"),
    limit=10,
    return_properties=[
        "title",
        "customer_type",
        "ai_summary",
        "ai_category",
        "ai_priority",
        "ai_action",
    ],
)

for obj in response.objects:
    print("Title:", obj.properties["title"])
    print("Customer type:", obj.properties["customer_type"])
    print("Category:", obj.properties["ai_category"])
    print("Priority:", obj.properties["ai_priority"])
    print("Action:", obj.properties["ai_action"])
    print("-" * 80)

Title: OpenAI key does not work in Weaviate Cloud
Customer type: developer
Category: other
Priority: high
Action: Check if the OpenAI API key is correctly configured and has the necessary permissions.
--------------------------------------------------------------------------------


In [16]:
response = tickets.query.hybrid(
    query="RAG notebooks chunking focused collection",
    alpha=0.5,
    limit=5,
    return_properties=[
        "title",
        "body",
        "ai_summary",
        "ai_category",
        "ai_priority",
        "ai_action",
    ],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Score:", obj.metadata.score)
    print("Title:", obj.properties["title"])
    print("Category:", obj.properties["ai_category"])
    print("Priority:", obj.properties["ai_priority"])
    print("Summary:", obj.properties["ai_summary"])
    print("Action:", obj.properties["ai_action"])
    print("-" * 80)

Score: 1.0
Title: Need help designing RAG over project notebooks
Category: rag
Priority: medium
Summary: Assistance required for designing a RAG collection from Jupyter notebooks and Markdown notes.
Action: Consider organizing your notes into categories to enhance chunk retrieval.
--------------------------------------------------------------------------------
Score: 0.08177244663238525
Title: Images are not searchable in Weaviate Cloud
Category: image_search
Priority: medium
Summary: Issues with searching images after migrating to Weaviate Cloud.
Action: Ensure that the CLIP embeddings are properly indexed in Weaviate Cloud.
--------------------------------------------------------------------------------
Score: 0.07628591358661652
Title: Objects are inserted but I do not understand UUIDs
Category: crud
Priority: medium
Summary: User is unclear on how to use UUIDs for fetching, updating, and deleting objects in Weaviate.
Action: Refer to the Weaviate documentation on CRUD operations us

In [17]:
def search_tickets(query: str, *, limit: int = 3) -> None:
    response = tickets.query.near_text(
        query=query,
        limit=limit,
        return_properties=[
            "title",
            "customer_type",
            "ai_summary",
            "ai_category",
            "ai_priority",
            "ai_action",
        ],
        return_metadata=MetadataQuery(distance=True),
    )

    print("QUERY:", query)
    print("-" * 80)

    for obj in response.objects:
        print("Distance:", obj.metadata.distance)
        print("Title:", obj.properties["title"])
        print("Customer type:", obj.properties["customer_type"])
        print("Category:", obj.properties["ai_category"])
        print("Priority:", obj.properties["ai_priority"])
        print("Summary:", obj.properties["ai_summary"])
        print("Action:", obj.properties["ai_action"])
        print("-" * 80)

In [18]:
search_tickets("OpenAI authentication error in Weaviate Cloud")

QUERY: OpenAI authentication error in Weaviate Cloud
--------------------------------------------------------------------------------
Distance: 0.28554606437683105
Title: OpenAI key does not work in Weaviate Cloud
Customer type: developer
Category: other
Priority: high
Summary: User is experiencing a 401 error when using OpenAI API key with Weaviate Cloud for data insertion.
Action: Check if the OpenAI API key is correctly configured and has the necessary permissions.
--------------------------------------------------------------------------------
Distance: 0.4405227303504944
Title: Images are not searchable in Weaviate Cloud
Customer type: ml_engineer
Category: image_search
Priority: medium
Summary: Issues with searching images after migrating to Weaviate Cloud.
Action: Ensure that the CLIP embeddings are properly indexed in Weaviate Cloud.
--------------------------------------------------------------------------------
Distance: 0.5521343946456909
Title: How to isolate data for multi

In [19]:
search_tickets("how to build RAG over notebooks")

QUERY: how to build RAG over notebooks
--------------------------------------------------------------------------------
Distance: 0.29967200756073
Title: Need help designing RAG over project notebooks
Customer type: data_scientist
Category: rag
Priority: medium
Summary: Assistance required for designing a RAG collection from Jupyter notebooks and Markdown notes.
Action: Consider organizing your notes into categories to enhance chunk retrieval.
--------------------------------------------------------------------------------
Distance: 0.8104149103164673
Title: How to isolate data for multiple companies
Customer type: product_engineer
Category: multi_tenancy
Priority: medium
Summary: Customer seeks guidance on isolating data for multiple companies using Weaviate's multi-tenancy feature.
Action: Refer to Weaviate's documentation on multi-tenancy implementation.
--------------------------------------------------------------------------------
Distance: 0.8303546905517578
Title: Objects are i

In [20]:
search_tickets("image search with CLIP embeddings")

QUERY: image search with CLIP embeddings
--------------------------------------------------------------------------------
Distance: 0.3754361867904663
Title: Images are not searchable in Weaviate Cloud
Customer type: ml_engineer
Category: image_search
Priority: medium
Summary: Issues with searching images after migrating to Weaviate Cloud.
Action: Ensure that the CLIP embeddings are properly indexed in Weaviate Cloud.
--------------------------------------------------------------------------------
Distance: 0.7696982622146606
Title: Hybrid search returns unexpected results
Customer type: developer
Category: hybrid_search
Priority: medium
Summary: Customer is experiencing unexpected results from hybrid search with alpha set to 0.5, noting a mix of relevant and keyword-based results.
Action: Review the impact of the alpha parameter on ranking and adjust accordingly.
--------------------------------------------------------------------------------
Distance: 0.7908684015274048
Title: OpenAI

In [21]:
search_tickets("isolated data for SaaS customers")

QUERY: isolated data for SaaS customers
--------------------------------------------------------------------------------
Distance: 0.52430260181427
Title: How to isolate data for multiple companies
Customer type: product_engineer
Category: multi_tenancy
Priority: medium
Summary: Customer seeks guidance on isolating data for multiple companies using Weaviate's multi-tenancy feature.
Action: Refer to Weaviate's documentation on multi-tenancy implementation.
--------------------------------------------------------------------------------
Distance: 0.7832281589508057
Title: Need help designing RAG over project notebooks
Customer type: data_scientist
Category: rag
Priority: medium
Summary: Assistance required for designing a RAG collection from Jupyter notebooks and Markdown notes.
Action: Consider organizing your notes into categories to enhance chunk retrieval.
--------------------------------------------------------------------------------
Distance: 0.7863644361495972
Title: Hybrid searc

In [22]:
response = tickets.generate.near_text(
    query="customer needs help with Weaviate Cloud and OpenAI API key authentication",
    grouped_task=(
        "Act as a Weaviate support assistant. "
        "Use only the retrieved support tickets. "
        "Summarize the likely issue and suggest a practical next step. "
        "Mention the detected category and priority if available."
    ),
    limit=3,
    return_properties=[
        "title",
        "body",
        "customer_type",
        "ai_summary",
        "ai_category",
        "ai_priority",
        "ai_action",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print("-", obj.properties["title"], "|", obj.properties["ai_category"], "|", obj.properties["ai_priority"])

### Ticket 1
**Likely Issue:** User is experiencing a 401 error when trying to insert data using the OpenAI API key in Weaviate Cloud, indicating that the API key may not be configured correctly or lacks the necessary permissions.

**Next Step:** Check if the OpenAI API key is correctly configured and has the necessary permissions.

**Category:** Other  
**Priority:** High  

---

### Ticket 2
**Likely Issue:** User is facing issues with searching images after migrating from a local Docker setup to Weaviate Cloud. The CLIP embeddings that worked previously may not be properly indexed.

**Next Step:** Ensure that the CLIP embeddings are properly indexed in Weaviate Cloud.

**Category:** Image Search  
**Priority:** Medium  

---

### Ticket 3
**Likely Issue:** Customer needs guidance on implementing multi-tenancy in Weaviate to isolate data for multiple companies in their SaaS product.

**Next Step:** Refer to Weaviate's documentation on multi-tenancy implementation for proper guidance.

In [23]:
client.close()